# 03 — Delivering and Evaluating a Grounded Solution

Uses the `state-driver-manuals` index from Module 2. `gpt-4.1-mini` serves the managed "on your data" call, because that extension still sends the legacy `max_tokens` parameter, which `gpt-5-mini` rejects.
`gpt-5-mini` runs the hand-built pipeline and the evaluators.


## Setup — run once per kernel session

In [ ]:
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery
from openai import AzureOpenAI

import sys
sys.path.insert(0, "..")

from config import load_config

config = load_config()
credential = DefaultAzureCredential()
search_client = SearchClient(config.search_endpoint, config.search_index_name, credential)

token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")
openai_client = AzureOpenAI(
    azure_endpoint=config.openai_endpoint,
    azure_ad_token_provider=token_provider,
    api_version="2024-10-21",
)

ONYOURDATA_DEPLOYMENT = config.openai_onyourdata_deployment  # gpt-4.1-mini
CHAT_DEPLOYMENT = config.openai_chat_deployment  # gpt-5-mini

HERO_QUESTION = "What's the speed limit in a school zone?"

print("Search index:", config.search_index_name)
print("On-your-data deployment:", ONYOURDATA_DEPLOYMENT)
print("Hand-rolled/judge deployment:", CHAT_DEPLOYMENT)


---
## Clip 1 (2 min) — Connecting the Index to a Model in Foundry

Attach the Search index as a data source on the chat call (Azure OpenAI "on your data"). Authentication is `system_assigned_managed_identity`: no API keys.


In [ ]:
def ask_with_managed_data_source(query, top_n=5):
    response = openai_client.chat.completions.create(
        model=ONYOURDATA_DEPLOYMENT,
        messages=[{"role": "user", "content": query}],
        extra_body={
            "data_sources": [{
                "type": "azure_search",
                "parameters": {
                    "endpoint": config.search_endpoint,
                    "index_name": config.search_index_name,
                    "authentication": {"type": "system_assigned_managed_identity"},
                    "query_type": "vector_semantic_hybrid",
                    "semantic_configuration": "semantic-config",
                    "embedding_dependency": {
                        "type": "deployment_name",
                        "deployment_name": config.openai_embedding_deployment,
                    },
                    "top_n_documents": top_n,
                },
            }],
        },
    )
    message = response.choices[0].message
    citations = (message.model_extra or {}).get("context", {}).get("citations", [])
    return message.content, citations


In [ ]:
answer, citations = ask_with_managed_data_source(HERO_QUESTION)
print("Answer:")
print(answer)
print()
print(f"{len(citations)} citation(s):")
for c in citations[:5]:
    print(f"  - {c.get('title')}: {c.get('content', '')[:150].strip()}...")


---
## Clip 2 (4 min) — Constructing the Grounded Prompt Pipeline

Rebuilt by hand so nothing is a black box: hybrid and semantic search, a system prompt that answers only from the passages, then a chat completion.


In [ ]:
import random
import threading
import time

from openai import RateLimitError

# evaluate() calls answer_question() concurrently; the semaphore caps parallel
# requests so the batch stays under the deployment's rate limit.
CHAT_SEMAPHORE = threading.Semaphore(3)

SYSTEM_PROMPT = (
    "Answer only using the passages in the context below. Cite the state "
    "each fact comes from. If the context does not contain the answer, say "
    "you don't have that information — do not use outside knowledge."
)


def retrieve_passages(query, state_filter=None, top=5):
    results = search_client.search(
        search_text=query,
        vector_queries=[VectorizableTextQuery(text=query, k_nearest_neighbors=top, fields="vector")],
        query_type="semantic",
        semantic_configuration_name="semantic-config",
        filter=f"state eq '{state_filter}'" if state_filter else None,
        top=top,
        select=["state", "title", "chunk"],
    )
    return [{"state": r["state"], "title": r["title"], "chunk": r["chunk"]} for r in results]


def format_context(passages):
    return "\n\n".join(f"[{p['state']}] {p['chunk']}" for p in passages)


def chat_completion_with_retry(**kwargs):
    with CHAT_SEMAPHORE:
        for attempt in range(8):
            try:
                return openai_client.chat.completions.create(**kwargs)
            except RateLimitError:
                if attempt == 7:
                    raise
                time.sleep(min(2 ** attempt, 30) + random.random())


def answer_question(query, state_filter=None, top=5):
    passages = retrieve_passages(query, state_filter=state_filter, top=top)
    context = format_context(passages)
    response = chat_completion_with_retry(
        model=CHAT_DEPLOYMENT,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
        ],
    )
    return {"answer": response.choices[0].message.content, "passages": passages, "context": context}


In [ ]:
result = answer_question(HERO_QUESTION)
print("Answer:", result["answer"])
print()
print("Retrieved from:", [p["state"] for p in result["passages"]])


**The refusal.** Texas isn't indexed; a pipeline that answers anyway is guessing, not grounded.


In [ ]:
refusal = answer_question("What's the school zone speed limit in Texas?")
print(refusal["answer"])


**The citation failure.** Ask a New Jersey question with the filter off and `top=1`.


In [ ]:
citation_failure = answer_question("How close to a fire hydrant can I park?", state_filter=None, top=1)
print(citation_failure["answer"])
print()
print("Actually retrieved from:", [p["state"] for p in citation_failure["passages"]])


---
## Clip 3 (4 min) — Measuring Groundedness and Relevance

Runs the 20-row eval set through `answer_question()` unfiltered, scored for retrieval, groundedness, and relevance.


In [ ]:
from azure.ai.evaluation import evaluate, RetrievalEvaluator, GroundednessEvaluator, RelevanceEvaluator

model_config = {
    "azure_endpoint": config.openai_endpoint,
    "azure_deployment": CHAT_DEPLOYMENT,
    # No `credential` key: the SDK's model_config validator rejects it.
    # The judge calls still authenticate keyless via DefaultAzureCredential.
}

# is_reasoning_model=True: gpt-5-mini rejects the legacy `max_tokens` parameter,
# so the evaluators send `max_completion_tokens` instead.
retrieval_evaluator = RetrievalEvaluator(model_config=model_config, is_reasoning_model=True)
groundedness_evaluator = GroundednessEvaluator(model_config=model_config, is_reasoning_model=True)
relevance_evaluator = RelevanceEvaluator(model_config=model_config, is_reasoning_model=True)


def eval_target(query, state, ground_truth):
    # evaluate() matches these parameter names to the data columns, so name each one.
    # state and ground_truth are unused: retrieval runs unfiltered on purpose.
    result = answer_question(query, state_filter=None, top=5)
    return {"response": result["answer"], "context": result["context"]}


In [ ]:
eval_result = evaluate(
    data="../eval/eval_set.jsonl",
    target=eval_target,
    evaluators={
        "retrieval": retrieval_evaluator,
        "groundedness": groundedness_evaluator,
        "relevance": relevance_evaluator,
    },
    evaluator_config={
        "retrieval": {"column_mapping": {"query": "${data.query}", "context": "${target.context}"}},
        "groundedness": {"column_mapping": {"query": "${data.query}", "response": "${target.response}", "context": "${target.context}"}},
        "relevance": {"column_mapping": {"query": "${data.query}", "response": "${target.response}"}},
    },
    output_path="../eval/eval_results.json",
    # If the Foundry project is not ready yet, only the portal upload is skipped.
    # Local scoring still runs.
    azure_ai_project=config.ai_project_endpoint or None,
)
print("Rows evaluated:", len(eval_result["rows"]))


In [ ]:
metrics = eval_result["metrics"]
for name, value in sorted(metrics.items()):
    print(f"{name}: {value}")


**Diagnose.** Take the lowest-retrieval row, re-run it with the state filter, and re-score it.


In [ ]:
import math

rows = eval_result["rows"]


def has_score(row):
    value = row.get("outputs.retrieval.retrieval")
    return isinstance(value, (int, float)) and not math.isnan(value)


scored = [r for r in rows if has_score(r)]
print("Rows without a retrieval score:", len(rows) - len(scored))

candidates = sorted(
    scored,
    key=lambda r: (r["outputs.retrieval.retrieval"], -r.get("outputs.groundedness.groundedness", 0)),
)
worst = candidates[0]
print("Lowest-retrieval row:")
print(" query:", worst.get("inputs.query"))
print(" expected state:", worst.get("inputs.state"))
print(" retrieval score:", worst.get("outputs.retrieval.retrieval"))
print(" groundedness score:", worst.get("outputs.groundedness.groundedness"))
print(" answer:", worst.get("outputs.response", "")[:200])


In [ ]:
# Re-run that one row with the state filter applied, then re-score it alone.
fixed = answer_question(worst["inputs.query"], state_filter=worst["inputs.state"], top=5)

retrieval_before = worst.get("outputs.retrieval.retrieval")
retrieval_after = retrieval_evaluator(query=worst["inputs.query"], context=fixed["context"])["retrieval"]
groundedness_after = groundedness_evaluator(query=worst["inputs.query"], response=fixed["answer"], context=fixed["context"])["groundedness"]

print(f"Retrieval score:    {retrieval_before} -> {retrieval_after}")
print(f"Groundedness score: {worst.get('outputs.groundedness.groundedness')} -> {groundedness_after}")
print()
print("Fixed answer:", fixed["answer"][:200])
